## 1. Install necessary libraries

In [ ]:
pip install faiss-cpu pymupdf transformers torch pillow numpy python-dotenv langchain langchain-openai langchain-text-splitters 

## 2. Import necessary libraries

In [11]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"


import faiss
import io
import base64
import numpy as np
import pymupdf  # PyMuPDF (the old name "fitz" still works, but "pymupdf" is the current one)
import torch
from PIL import Image
from dotenv import load_dotenv
from transformers import CLIPModel, CLIPProcessor

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)  # reads OPENAI_API_KEY from your .env file



True

## 3. Load CLIP Model

In [2]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 32151.40it/s]


## 4. Embedding functions

In [3]:
def embed_image(image):
    """Embed a PIL image with CLIP."""
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs).pooler_output
    features = features / features.norm(dim=-1, keepdim=True)  # unit length
    return features[0].numpy()


def embed_text(text):
    """Embed text with CLIP."""
    inputs = clip_processor(text=text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs).pooler_output
    features = features / features.norm(dim=-1, keepdim=True)
    return features[0].numpy()

In [4]:
red = embed_image(Image.new("RGB", (224, 224), "red"))
print(embed_text("a chart showing revenue").shape, red.shape)
print("red:", red @ embed_text("a solid red image"), "| dog:", red @ embed_text("a photo of a dog"))

(512,) (512,)
red: 0.31685418 | dog: 0.23687184


## 2. Extract PDF text and images

In [5]:
pdf_path = "multimodal_sample.pdf"  # must be in the same folder as your notebook

all_docs = []          # text chunks and image placeholders
all_embeddings = []    # one CLIP vector per item in all_docs
image_data_store = {}  # image_id -> base64 string (sent to GPT later)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

with pymupdf.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf):

        # 1) Text: split into chunks, embed each chunk
        text = page.get_text() 
        if text.strip(): #skips pages with no text (an image-only page, for example)
            page_doc = Document(page_content=text, metadata={"page": page_num, "type": "text"})
            for chunk in splitter.split_documents([page_doc]):
                all_docs.append(chunk)
                all_embeddings.append(embed_text(chunk.page_content))

        # 2) Images: convert, store as base64, embed
        for img_num, img in enumerate(page.get_images(full=True)):
            try:
                image_bytes = pdf.extract_image(img[0])["image"]
                pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

                image_id = f"page_{page_num}_img_{img_num}"

                buffer = io.BytesIO()
                pil_image.save(buffer, format="PNG")
                image_data_store[image_id] = base64.b64encode(buffer.getvalue()).decode()

                all_docs.append(Document(
                    page_content=f"[Image: {image_id}]",
                    metadata={"page": page_num, "type": "image", "image_id": image_id},
                ))
                all_embeddings.append(embed_image(pil_image))
            except Exception as e:
                print(f"Skipped image {img_num} on page {page_num}: {e}")

# Check the results
embeddings_array = np.array(all_embeddings)

print("Total items:", len(all_docs))
print("Text chunks:", sum(d.metadata["type"] == "text" for d in all_docs))
print("Images:", len(image_data_store))
print("Embeddings shape:", embeddings_array.shape)

Total items: 2
Text chunks: 1
Images: 1
Embeddings shape: (2, 512)


In [6]:
index = faiss.IndexFlatIP(512)                  # empty index for 512-number vectors
index.add(embeddings_array.astype("float32"))   # put all our vectors inside

print("Vectors in index:", index.ntotal)
def retrieve_multimodal(query, k=5):
    """Find the k text chunks or images most similar to the query."""
    query_embedding = embed_text(query).astype("float32")
    scores, ids = index.search(query_embedding[None, :], k)
    return [all_docs[i] for i in ids[0]]


results = retrieve_multimodal("What does the chart show about revenue?", k=5)
for doc in results:
    print(doc.metadata["type"], "| page", doc.metadata["page"], "|", doc.page_content[:80])    

Vectors in index: 2
text | page 0 | Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q
image | page 0 | [Image: page_0_img_0]
image | page 0 | [Image: page_0_img_0]
image | page 0 | [Image: page_0_img_0]
image | page 0 | [Image: page_0_img_0]


In [7]:
llm = init_chat_model(model="gpt-5-nano", temperature=0)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.1', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x179897910>, async_client=<openai.resources.

In [8]:
def create_multimodal_message(query, retrieved_docs):
    """Put the question, text chunks and images into one message for GPT."""
    content = [{"type": "text", "text": f"Question: {query}\n\nContext:\n"}]

    for doc in retrieved_docs:
        if doc.metadata["type"] == "text":
            content.append({
                "type": "text",
                "text": f"[Text from page {doc.metadata['page']}]: {doc.page_content}\n",
            })
        else:
            image_b64 = image_data_store[doc.metadata["image_id"]]
            content.append({"type": "text", "text": f"[Image from page {doc.metadata['page']}]:"})
            content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"},
            })

    content.append({"type": "text", "text": "\nAnswer the question using the text and images above."})
    return HumanMessage(content=content)

In [9]:
def multimodal_pdf_rag_pipeline(query):
    docs = retrieve_multimodal(query, k=5)
    message = create_multimodal_message(query, docs)
    response = llm.invoke([message])

    print(f"Retrieved {len(docs)} items:")
    for doc in docs:
        print(f"  - {doc.metadata['type']} from page {doc.metadata['page']}")

    return response.content

In [12]:
queries = [
    "What does the chart show about revenue trends?",
    "Summarize the main findings from the document",
]

for query in queries:
    print(f"\nQuery: {query}")
    print(multimodal_pdf_rag_pipeline(query))
    print("=" * 60)


Query: What does the chart show about revenue trends?


OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************qoEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}